# Notebook 1: Data Loading

### StudentLife Dataset — Spring 2013
Load raw sensor, EMA, and survey data for the relevant participants.
All location data is dropped immediately on load for privacy.

In [47]:
import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

# Base path to dataset
DATA_DIR = Path("../data/studentlife")

# Study period
STUDY_START = pd.Timestamp("2013-03-25", tz="US/Eastern")
STUDY_END   = pd.Timestamp("2013-06-03", tz="US/Eastern")

# All participant IDs
UIDS = [f"u{i:02d}" for i in range(48)]

print("Imports complete.")
print(f"Study period: {STUDY_START.date()} to {STUDY_END.date()}")
print(f"Participants: {len(UIDS)}")

Imports complete.
Study period: 2013-03-25 to 2013-06-03
Participants: 48


In [48]:
all_pam_dates = []
for uid in [f"u{i:02d}" for i in range(48)]:
    fpath = DATA_DIR / "EMA" / "response" / "PAM" / f"PAM_{uid}.json"
    if not fpath.exists():
        continue
    with open(fpath) as f:
        entries = json.load(f)
    for entry in entries:
        resp_time = entry.get("resp_time")
        if resp_time:
            dt = pd.to_datetime(resp_time, unit="s", utc=True).tz_convert("US/Eastern")
            all_pam_dates.append(dt.date())

all_pam_dates = pd.to_datetime(all_pam_dates)
print(f"PAM date range: {all_pam_dates.min().date()} to {all_pam_dates.max().date()}")
print(f"Unique dates: {all_pam_dates.nunique()}")

PAM date range: 2013-03-24 to 2013-07-13
Unique dates: 78


In [49]:
import os
import json
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

# ─────────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────────

DATA_DIR      = Path("../data/studentlife")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# STUDY PARAMETERS
# ─────────────────────────────────────────────────────────────────

# Official Dartmouth Spring 2013 term dates
# Source: Dartmouth Office of the Registrar Spring 2013 Term Calendar
STUDY_START = pd.Timestamp("2013-03-25", tz="US/Eastern")
STUDY_END   = pd.Timestamp("2013-06-03", tz="US/Eastern")

# Actual participant UIDs present in the dataset
# NOTE: IDs are NOT sequential 0-48. Several numbers are skipped
# (u06, u11, u21, u26, u28, u29, u37, u38, u40, u48, u55 do not exist)
# These were derived by listing the actual files in the sensing/activity folder
UIDS = [
    "u00","u01","u02","u03","u04","u05","u07","u08","u09","u10",
    "u12","u13","u14","u15","u16","u17","u18","u19","u20","u22",
    "u23","u24","u25","u27","u30","u31","u32","u33","u34","u35",
    "u36","u39","u41","u42","u43","u44","u45","u46","u47","u49",
    "u50","u51","u52","u53","u54","u56","u57","u58","u59"
]

# ─────────────────────────────────────────────────────────────────
# PRIVACY: UID HASHING
# ─────────────────────────────────────────────────────────────────

# All participant UIDs are irreversibly hashed before any data is stored.
# We use SHA-256 with a salt string. The raw UID is never passed downstream.
# This satisfies GDPR Art. 25 (privacy by design / data minimisation).

def hash_uid(uid, salt="ddcm_2024"):
    return hashlib.sha256(f"{salt}:{uid}".encode()).hexdigest()[:16]

uid_map = {uid: hash_uid(uid) for uid in UIDS}

print(f"Participants: {len(UIDS)}")
print(f"Study period: {STUDY_START.date()} to {STUDY_END.date()}")
print(f"Example hash: u00 -> {uid_map['u00']}")

# ─────────────────────────────────────────────────────────────────
# PAM (PHOTOGRAPHIC AFFECT METER) — PRIMARY MOOD OUTCOME
# ─────────────────────────────────────────────────────────────────

# PAM is a validated in-situ measure of affect where students select
# one of 16 photos arranged in a 4x4 grid representing Russell's
# two-dimensional affect space (valence x arousal).
# It was fired before every other EMA prompt in StudentLife, guaranteeing
# high compliance: 6,660 entries across 39 participants vs 137 for Mood EMA.
#
# Validation: PAM correlates with PANAS Positive Affect at r=0.71 (p<0.001)
# Source: Pollak, Adams & Gay (2011). PAM: A Photographic Affect Meter
#         for Frequent, In Situ Measurement of Affect. CHI 2011.
#
# Scoring: The raw picture_idx (1-16) is used as the analytical variable.
# Higher scores = higher positive affect. For presentation purposes only,
# scores map to four quadrants from Figure 3 of Pollak et al. (2011):
#   1-4  = Negative valence / Low arousal  (sad, depressed, bored)
#   5-8  = Negative valence / High arousal (stressed, anxious, angry)
#   9-12 = Positive valence / Low arousal  (calm, content, relaxed)
#   13-16= Positive valence / High arousal (excited, happy, enthusiastic)
#
# We take the daily MEAN PAM score per participant since PAM fires
# multiple times per day (once before each EMA prompt).
# Location data is not collected by PAM — no privacy filtering needed.

pam_records = []

for uid in UIDS:
    fpath = DATA_DIR / "EMA" / "response" / "PAM" / f"PAM_{uid}.json"
    if not fpath.exists():
        continue

    with open(fpath) as f:
        entries = json.load(f)

    for entry in entries:
        try:
            picture_idx = entry.get("picture_idx")
            resp_time   = entry.get("resp_time")
            if picture_idx is None or resp_time is None:
                continue
            pam_records.append({
                "hashed_uid": uid_map[uid],
                "resp_time":  resp_time,
                "pam_score":  int(picture_idx)   # raw 1-16 score
            })
        except (TypeError, ValueError):
            continue

df_pam = pd.DataFrame(pam_records)
df_pam["datetime"] = pd.to_datetime(df_pam["resp_time"], unit="s", utc=True).dt.tz_convert("US/Eastern")
df_pam["date"]     = df_pam["datetime"].dt.normalize()

# Filter to study period
df_pam = df_pam[
    (df_pam["date"] >= STUDY_START.normalize()) &
    (df_pam["date"] <= STUDY_END.normalize())
]

# Aggregate to daily mean — multiple PAM responses per day are expected
df_pam = (df_pam
          .groupby(["hashed_uid", "date"])["pam_score"]
          .agg(pam_score_mean="mean", pam_n_responses="count")
          .reset_index())

print(f"\nPAM loaded: {len(df_pam)} participant-days, {df_pam['hashed_uid'].nunique()} participants")

# ─────────────────────────────────────────────────────────────────
# ACTIVITY — PASSIVE SENSOR (behavioral proxy for movement)
# ─────────────────────────────────────────────────────────────────

# The StudentLife activity classifier runs 24/7 and generates an
# activity inference every 2-3 seconds based on accelerometer data.
# Activity codes: 0=stationary, 1=walking, 2=running, 3=unknown
# Source: Wang et al. (2014) StudentLife, UbiComp 2014.
#
# We aggregate to daily level: fraction of readings that are stationary (code=0)
# Higher frac_stationary = less physical movement that day
# This serves as a passive behavioral proxy for physical activity level.

activity_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "activity" / f"activity_{uid}.csv"
    if not fpath.exists():
        continue

    df = pd.read_csv(fpath, header=0, names=["timestamp", "activity"])
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("US/Eastern")
    df["date"]     = df["datetime"].dt.date

    # Aggregate to daily: fraction of readings that are stationary
    daily = df.groupby("date").apply(
        lambda g: pd.Series({
            "frac_stationary":     (g["activity"] == 0).mean(),
            "n_activity_readings": len(g)
        })
    ).reset_index()

    daily["hashed_uid"] = uid_map[uid]
    activity_records.append(daily)

df_activity = pd.concat(activity_records, ignore_index=True)
df_activity["date"] = pd.to_datetime(df_activity["date"])
df_activity = df_activity[
    (df_activity["date"] >= pd.Timestamp("2013-03-25")) &
    (df_activity["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Activity loaded: {len(df_activity)} participant-days")

# ─────────────────────────────────────────────────────────────────
# DARK PERIODS — PASSIVE SENSOR (sleep/inactivity proxy)
# ─────────────────────────────────────────────────────────────────

# The dark sensor records start and end Unix timestamps of periods
# when the phone screen was continuously dark/off.
# We filter to periods >= 1 hour to remove brief screen-off moments
# (e.g. pocket, table) and retain only sustained inactivity periods
# that are more likely to correspond to sleep or extended rest.
#
# We sum total dark hours per day as a proxy for sleep duration.
# This is preferred over phonelock (which fires constantly throughout
# the day) because dark periods are longer and more continuous,
# aligning better with actual sleep windows.
# Note: Wang et al. (2014) used a dedicated sleep classifier combining
# light, phone lock, activity and audio features. The dark sensor alone
# is a simplified but reasonable proxy for our purposes.

dark_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "dark" / f"dark_{uid}.csv"
    if not fpath.exists():
        continue

    df = pd.read_csv(fpath, header=0, names=["start_ts", "end_ts"])
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df["duration_hours"] = (df["end_ts"] - df["start_ts"]) / 3600

    # Keep only periods >= 1 hour (filters brief screen-off moments)
    df = df[df["duration_hours"] >= 1]

    df["date"] = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("US/Eastern").dt.date

    daily = df.groupby("date")["duration_hours"].sum().reset_index()
    daily.columns = ["date", "dark_hours"]
    daily["hashed_uid"] = uid_map[uid]
    dark_records.append(daily)

df_dark = pd.concat(dark_records, ignore_index=True)
df_dark["date"] = pd.to_datetime(df_dark["date"])
df_dark = df_dark[
    (df_dark["date"] >= pd.Timestamp("2013-03-25")) &
    (df_dark["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Dark periods loaded: {len(df_dark)} participant-days")

# ─────────────────────────────────────────────────────────────────
# CONVERSATION — PASSIVE SENSOR (social activity proxy)
# ─────────────────────────────────────────────────────────────────

# The StudentLife conversation classifier uses the phone microphone
# to detect periods of conversation (human voice around the participant).
# IMPORTANT: No audio content is ever recorded or stored. The classifier
# only infers whether conversation is occurring — not what is said.
# Output: start and end timestamps of detected conversation periods.
#
# We sum total conversation hours per day as a proxy for social activity.
# Wang et al. (2014) found significant negative correlations between
# conversation frequency/duration and PHQ-9 depression scores.

conversation_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "conversation" / f"conversation_{uid}.csv"
    if not fpath.exists():
        continue

    df = pd.read_csv(fpath, header=0, names=["start_ts", "end_ts"])
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    df["duration_hours"] = (df["end_ts"] - df["start_ts"]) / 3600
    df["date"] = pd.to_datetime(df["start_ts"], unit="s", utc=True).dt.tz_convert("US/Eastern").dt.date

    daily = df.groupby("date")["duration_hours"].sum().reset_index()
    daily.columns = ["date", "conversation_hours"]
    daily["hashed_uid"] = uid_map[uid]
    conversation_records.append(daily)

df_conversation = pd.concat(conversation_records, ignore_index=True)
df_conversation["date"] = pd.to_datetime(df_conversation["date"])
df_conversation = df_conversation[
    (df_conversation["date"] >= pd.Timestamp("2013-03-25")) &
    (df_conversation["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"Conversation loaded: {len(df_conversation)} participant-days")

# ─────────────────────────────────────────────────────────────────
# GPS — PASSIVE SENSOR (mobility proxy)
# ─────────────────────────────────────────────────────────────────

# GPS coordinates are collected passively throughout the day.
# PRIVACY: Raw latitude/longitude are NEVER stored downstream.
# We derive daily distance from Dartmouth campus center (Hanover NH)
# as a mobility proxy using the Haversine formula.
# Readings with accuracy > 100 meters are filtered out — these are
# likely cell tower estimates rather than true GPS fixes and would
# produce unreliable distance calculations.
# Campus center coordinates: 43.7044, -72.2887
# Source: Dartmouth College, Hanover NH

DARTMOUTH_LAT = 43.7044
DARTMOUTH_LON = -72.2887

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance in km between two coordinates."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

gps_records = []

for uid in UIDS:
    fpath = DATA_DIR / "sensing" / "gps" / f"gps_{uid}.csv"
    if not fpath.exists():
        continue

    df = pd.read_csv(fpath, index_col=False)
    for col in ["time", "latitude", "longitude", "accuracy"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["time", "latitude", "longitude"])

    # Filter to high-accuracy GPS readings only (<=100m)
    # Readings above this threshold are likely cell tower estimates
    if "accuracy" in df.columns:
        df = df[df["accuracy"] <= 100]

    if df.empty:
        continue

    df["datetime"] = pd.to_datetime(df["time"], unit="s", utc=True).dt.tz_convert("US/Eastern")
    df["date"]     = df["datetime"].dt.date

    daily = df.groupby("date").agg(
        latitude        =("latitude", "median"),
        longitude       =("longitude", "median"),
        n_gps_readings  =("time", "size"),
        mean_accuracy   =("accuracy", "mean")
    ).reset_index()

    if daily.empty:
        continue

    # Compute distance from campus then drop raw coordinates immediately
    daily["km_from_campus"] = daily.apply(
        lambda row: haversine_km(
            DARTMOUTH_LAT, DARTMOUTH_LON,
            row["latitude"], row["longitude"]
        ), axis=1
    )
    daily = daily.drop(columns=["latitude", "longitude"])
    daily["hashed_uid"] = uid_map[uid]
    gps_records.append(daily)

if gps_records:
    df_gps = pd.concat(gps_records, ignore_index=True)
else:
    df_gps = pd.DataFrame(columns=[
        "date", "n_gps_readings", "mean_accuracy", "km_from_campus", "hashed_uid"
    ])

df_gps["date"] = pd.to_datetime(df_gps["date"])
df_gps = df_gps[
    (df_gps["date"] >= pd.Timestamp("2013-03-25")) &
    (df_gps["date"] <= pd.Timestamp("2013-06-03"))
]

print(f"GPS loaded: {len(df_gps)} participant-days")

# ─────────────────────────────────────────────────────────────────
# DEADLINES — CONTEXTUAL CONFOUND (academic pressure)
# ─────────────────────────────────────────────────────────────────

# The education/deadlines.csv file contains a per-participant,
# per-day count of assignment/exam deadlines.
# This is a direct, participant-specific measure of academic pressure —
# far superior to hardcoded exam week flags because it captures
# individual differences in workload (not all students have the same
# deadlines on the same days, except for the shared CS65 class).
# Values are 0 (no deadline) or a positive integer (number of deadlines).
# Source: StudentLife dataset — collected via exit interviews and
#         validated against students' calendars and returned assignments.

df_deadlines_raw = pd.read_csv(DATA_DIR / "education" / "deadlines.csv")

# The CSV is wide format: uid in first column, then one column per date
# We melt to long format: one row per participant per day
df_deadlines = df_deadlines_raw.melt(id_vars="uid", var_name="date", value_name="n_deadlines")
df_deadlines["date"]       = pd.to_datetime(df_deadlines["date"])
df_deadlines["hashed_uid"] = df_deadlines["uid"].map(uid_map)
df_deadlines = df_deadlines[["hashed_uid", "date", "n_deadlines"]]

# Filter to study period
df_deadlines = df_deadlines[
    (df_deadlines["date"] >= pd.Timestamp("2013-03-25")) &
    (df_deadlines["date"] <= pd.Timestamp("2013-06-03"))
]

# Binary flag: did this participant have any deadline today?
df_deadlines["has_deadline"] = (df_deadlines["n_deadlines"] > 0).astype(int)

print(f"Deadlines loaded: {len(df_deadlines)} participant-days")
print(f"  Days with at least 1 deadline: {df_deadlines['has_deadline'].sum()}")

# ─────────────────────────────────────────────────────────────────
# SAVE ALL PROCESSED FILES
# ─────────────────────────────────────────────────────────────────

df_pam.to_csv(PROCESSED_DIR / "pam.csv", index=False)
df_activity.to_csv(PROCESSED_DIR / "activity.csv", index=False)
df_dark.to_csv(PROCESSED_DIR / "dark.csv", index=False)
df_conversation.to_csv(PROCESSED_DIR / "conversation.csv", index=False)
df_gps.to_csv(PROCESSED_DIR / "gps.csv", index=False)
df_deadlines.to_csv(PROCESSED_DIR / "deadlines.csv", index=False)

print("\nAll files saved to data/processed/")
print("\nRow counts:")
for name, df in [("pam",          df_pam),
                 ("activity",     df_activity),
                 ("dark",         df_dark),
                 ("conversation", df_conversation),
                 ("gps",          df_gps),
                 ("deadlines",    df_deadlines)]:
    print(f"  {name}: {len(df)} rows")

Participants: 49
Study period: 2013-03-25 to 2013-06-03
Example hash: u00 -> cd6251b4de52453b

PAM loaded: 2251 participant-days, 49 participants
Activity loaded: 2866 participant-days
Dark periods loaded: 2360 participant-days
Conversation loaded: 2700 participant-days
GPS loaded: 2813 participant-days
Deadlines loaded: 3036 participant-days
  Days with at least 1 deadline: 641

All files saved to data/processed/

Row counts:
  pam: 2251 rows
  activity: 2866 rows
  dark: 2360 rows
  conversation: 2700 rows
  gps: 2813 rows
  deadlines: 3036 rows


In [50]:
df_pam.head()

,hashed_uid,date,pam_score_mean,pam_n_responses
0,02b34100f41b4332,2013-03-27 00:00:00-04:00,9.40,5
1,02b34100f41b4332,2013-03-28 00:00:00-04:00,10.25,4
2,02b34100f41b4332,2013-03-29 00:00:00-04:00,6.00,3
3,02b34100f41b4332,2013-03-30 00:00:00-04:00,9.20,5
4,02b34100f41b4332,2013-03-31 00:00:00-04:00,7.00,2


In [53]:
# Check what actually got saved
df_gps_check = pd.read_csv(PROCESSED_DIR / "gps.csv")
print(df_gps_check.columns.tolist())
print(df_gps_check.head())

['date', 'n_gps_readings', 'mean_accuracy', 'km_from_campus', 'hashed_uid']
         date  n_gps_readings  mean_accuracy  km_from_campus        hashed_uid
0  2013-03-27              49      25.138306        0.489856  cd6251b4de52453b
1  2013-03-28              42      32.410405        0.280354  cd6251b4de52453b
2  2013-03-29              49      30.486204        0.380112  cd6251b4de52453b
3  2013-03-30              41      28.073171        6.902603  cd6251b4de52453b
4  2013-03-31              68      21.176471        6.902932  cd6251b4de52453b
